In [3]:
#Bài 3. Cài đặt thuật toán minimax cho bài toán tictactoe với ma trận 5x5, 10x10, NxN.
def evaluate(board_state, player):
    if board_state.winner == player: # AI wins
        return 10
    elif board_state.winner is not None: # Opponent wins
        return -10
    return 0 # Draw

def minimax(board_state, depth, is_maximizing_player, ai_player, opponent_player):
    if board_state.is_game_over():
        return evaluate(board_state, ai_player)

    if is_maximizing_player:
        max_eval = -float('inf')
        for r, c in board_state.get_empty_cells():
            # Simulate move
            board_copy = TicTacToe(board_state.size) # Create a fresh copy to avoid side effects
            board_copy.board = [row[:] for row in board_state.board]
            board_copy.current_player = board_state.current_player
            board_copy.empty_cells = board_state.empty_cells
            board_copy.winner = board_state.winner

            board_copy.make_move(r, c)
            eval = minimax(board_copy, depth + 1, False, ai_player, opponent_player)
            max_eval = max(max_eval, eval)
        return max_eval
    else:
        min_eval = float('inf')
        for r, c in board_state.get_empty_cells():
            # Simulate move
            board_copy = TicTacToe(board_state.size)
            board_copy.board = [row[:] for row in board_state.board]
            board_copy.current_player = board_state.current_player
            board_copy.empty_cells = board_state.empty_cells
            board_copy.winner = board_state.winner

            board_copy.make_move(r, c)
            eval = minimax(board_copy, depth + 1, True, ai_player, opponent_player)
            min_eval = min(min_eval, eval)
        return min_eval

def find_best_move(board_state, ai_player, opponent_player):
    best_eval = -float('inf')
    best_move = None

    for r, c in board_state.get_empty_cells():
        board_copy = TicTacToe(board_state.size)
        board_copy.board = [row[:] for row in board_state.board]
        board_copy.current_player = board_state.current_player
        board_copy.empty_cells = board_state.empty_cells
        board_copy.winner = board_state.winner

        board_copy.make_move(r, c) # AI makes a move
        eval = minimax(board_copy, 0, False, ai_player, opponent_player)

        if eval > best_eval:
            best_eval = eval
            best_move = (r, c)
    return best_move


### Test Minimax with a simple game

Let's integrate the Minimax algorithm into a simple game loop to see our AI play.

In [4]:
class TicTacToe:
    def __init__(self, size):
        if not isinstance(size, int) or size < 3:
            raise ValueError("Board size must be an integer greater than or equal to 3.")
        self.size = size
        self.board = [[' ' for _ in range(size)] for _ in range(size)]
        self.current_player = 'X'
        self.winner = None
        self.empty_cells = size * size

    def print_board(self):
        for row in self.board:
            print('| ' + ' | '.join(row) + ' |')
            print('-' * (4 * self.size + 1))

    def is_valid_move(self, row, col):
        return 0 <= row < self.size and 0 <= col < self.size and self.board[row][col] == ' '

    def make_move(self, row, col):
        if not self.is_valid_move(row, col):
            return False
        self.board[row][col] = self.current_player
        self.empty_cells -= 1
        if self._check_win(row, col):
            self.winner = self.current_player
        else:
            self.current_player = 'O' if self.current_player == 'X' else 'X'
        return True

    def _check_win(self, r, c):
        player = self.board[r][c]

        # Check row
        if all(self.board[r][col] == player for col in range(self.size)):
            return True
        # Check column
        if all(self.board[row][c] == player for row in range(self.size)):
            return True
        # Check main diagonal
        if r == c and all(self.board[i][i] == player for i in range(self.size)):
            return True
        # Check anti-diagonal
        if r + c == self.size - 1 and all(self.board[i][self.size - 1 - i] == player for i in range(self.size)):
            return True
        return False

    def is_game_over(self):
        return self.winner is not None or self.empty_cells == 0

    def get_empty_cells(self):
        cells = []
        for r in range(self.size):
            for c in range(self.size):
                if self.board[r][c] == ' ':
                    cells.append((r, c))
        return cells

game_ai = TicTacToe(3)
print("Initial 3x3 board:")
game_ai.print_board()

player_X = 'X'
player_O = 'O'

while not game_ai.is_game_over():
    if game_ai.current_player == player_X:
        # Human player input
        print(f"Player {game_ai.current_player}'s turn.")
        try:
            move_row = int(input("Enter row (0-2): "))
            move_col = int(input("Enter column (0-2): "))
            if not game_ai.make_move(move_row, move_col):
                print("Invalid move, try again.")
                continue
        except ValueError:
            print("Invalid input. Please enter numbers.")
            continue
    else:
        # AI player
        print(f"Player {game_ai.current_player}'s (AI) turn.")
        ai_move = find_best_move(game_ai, player_O, player_X)
        if ai_move:
            game_ai.make_move(ai_move[0], ai_move[1])
        else:
            print("AI could not find a move. This should not happen in a non-full board.")
            break

    game_ai.print_board()
    if game_ai.winner:
        print(f"Game Over! Winner: {game_ai.winner}")
        break
    elif game_ai.empty_cells == 0:
        print("Game Over! It's a draw.")
        break

print("Game finished.")


Initial 3x3 board:
|   |   |   |
-------------
|   |   |   |
-------------
|   |   |   |
-------------
Player X's turn.
Enter row (0-2): 1
Enter column (0-2): 1
|   |   |   |
-------------
|   | X |   |
-------------
|   |   |   |
-------------
Player O's (AI) turn.
| O |   |   |
-------------
|   | X |   |
-------------
|   |   |   |
-------------
Player X's turn.
Enter row (0-2): 0
Enter column (0-2): 3
Invalid move, try again.
Player X's turn.
Enter row (0-2): 2
Enter column (0-2): 1
| O |   |   |
-------------
|   | X |   |
-------------
|   | X |   |
-------------
Player O's (AI) turn.
| O | O |   |
-------------
|   | X |   |
-------------
|   | X |   |
-------------
Player X's turn.
Enter row (0-2): 2
Enter column (0-2): 0
| O | O |   |
-------------
|   | X |   |
-------------
| X | X |   |
-------------
Player O's (AI) turn.
| O | O | O |
-------------
|   | X |   |
-------------
| X | X |   |
-------------
Game Over! Winner: O
Game finished.


In [6]:
#Bài 4. Cài đặt thuật toán cắt tỉa Alpha-beta cho bài toán tictactoe với ma trận 5x5, 10x10, NxN.

def evaluate_alpha_beta(board_state, player):
    if board_state.winner == player: # AI wins
        return 10
    elif board_state.winner is not None: # Opponent wins
        return -10
    return 0 # Draw

def minimax_alpha_beta(board_state, depth, alpha, beta, is_maximizing_player, ai_player, opponent_player):
    if board_state.is_game_over():
        return evaluate_alpha_beta(board_state, ai_player)

    if is_maximizing_player:
        max_eval = -float('inf')
        for r, c in board_state.get_empty_cells():
            board_copy = TicTacToe(board_state.size)
            board_copy.board = [row[:] for row in board_state.board]
            board_copy.current_player = board_state.current_player
            board_copy.empty_cells = board_state.empty_cells
            board_copy.winner = board_state.winner

            board_copy.make_move(r, c)
            eval = minimax_alpha_beta(board_copy, depth + 1, alpha, beta, False, ai_player, opponent_player)
            max_eval = max(max_eval, eval)
            alpha = max(alpha, eval)
            if beta <= alpha:
                break # Beta Cut-off
        return max_eval
    else:
        min_eval = float('inf')
        for r, c in board_state.get_empty_cells():
            board_copy = TicTacToe(board_state.size)
            board_copy.board = [row[:] for row in board_state.board]
            board_copy.current_player = board_state.current_player
            board_copy.empty_cells = board_state.empty_cells
            board_copy.winner = board_state.winner

            board_copy.make_move(r, c)
            eval = minimax_alpha_beta(board_copy, depth + 1, alpha, beta, True, ai_player, opponent_player)
            min_eval = min(min_eval, eval)
            beta = min(beta, eval)
            if beta <= alpha:
                break # Alpha Cut-off
        return min_eval

def find_best_move_alpha_beta(board_state, ai_player, opponent_player):
    best_eval = -float('inf')
    best_move = None

    alpha = -float('inf')
    beta = float('inf')

    for r, c in board_state.get_empty_cells():
        board_copy = TicTacToe(board_state.size)
        board_copy.board = [row[:] for row in board_state.board]
        board_copy.current_player = board_state.current_player
        board_copy.empty_cells = board_state.empty_cells
        board_copy.winner = board_state.winner

        board_copy.make_move(r, c) # AI makes a move
        eval = minimax_alpha_beta(board_copy, 0, alpha, beta, False, ai_player, opponent_player)

        if eval > best_eval:
            best_eval = eval
            best_move = (r, c)
        alpha = max(alpha, eval)
    return best_move

In [8]:
game_alpha_beta = TicTacToe(3)
print("Initial 3x3 board (Alpha-Beta AI):")
game_alpha_beta.print_board()

player_X = 'X'
player_O = 'O'

while not game_alpha_beta.is_game_over():
    if game_alpha_beta.current_player == player_X:
        # Human player input
        print(f"Player {game_alpha_beta.current_player}'s turn.")
        try:
            move_row = int(input("Enter row (0-2): "))
            move_col = int(input("Enter column (0-2): "))
            if not game_alpha_beta.make_move(move_row, move_col):
                print("Invalid move, try again.")
                continue
        except ValueError:
            print("Invalid input. Please enter numbers.")
            continue
    else:
        # AI player
        print(f"Player {game_alpha_beta.current_player}'s (Alpha-Beta AI) turn.")
        ai_move = find_best_move_alpha_beta(game_alpha_beta, player_O, player_X)
        if ai_move:
            game_alpha_beta.make_move(ai_move[0], ai_move[1])
        else:
            print("AI could not find a move. This should not happen in a non-full board.")
            break

    game_alpha_beta.print_board()
    if game_alpha_beta.winner:
        print(f"Game Over! Winner: {game_alpha_beta.winner}")
        break
    elif game_alpha_beta.empty_cells == 0:
        print("Game Over! It's a draw.")
        break

print("Game finished (Alpha-Beta AI).")

Initial 3x3 board (Alpha-Beta AI):
|   |   |   |
-------------
|   |   |   |
-------------
|   |   |   |
-------------
Player X's turn.
Enter row (0-2): 1
Enter column (0-2): 0
|   |   |   |
-------------
| X |   |   |
-------------
|   |   |   |
-------------
Player O's (Alpha-Beta AI) turn.
| O |   |   |
-------------
| X |   |   |
-------------
|   |   |   |
-------------
Player X's turn.
Enter row (0-2): 2
Enter column (0-2): 0
| O |   |   |
-------------
| X |   |   |
-------------
| X |   |   |
-------------
Player O's (Alpha-Beta AI) turn.
| O | O |   |
-------------
| X |   |   |
-------------
| X |   |   |
-------------
Player X's turn.
Enter row (0-2): 0
Enter column (0-2): 2
| O | O | X |
-------------
| X |   |   |
-------------
| X |   |   |
-------------
Player O's (Alpha-Beta AI) turn.
| O | O | X |
-------------
| X | O |   |
-------------
| X |   |   |
-------------
Player X's turn.
Enter row (0-2): 0
Enter column (0-2): 1
Invalid move, try again.
Player X's turn.
Ente